In [0]:
%run ../gold/00_gold_helpers

In [0]:
logger = get_logger("gold_customer_sales")

try:

    logger.info("Starting Gold Customer Sales transformation")

    # ---------------------------------------------------------
    # Read Customers
    # ---------------------------------------------------------

    logger.info("Reading Silver Customers table")

    df1 = read_table("customers_clean")

    logger.info("Silver Customers table read successfully")

    logger.info(
        "Removing technical columns from Customers"
    )

    df1 = df1.drop(
        "_ingestion_timestamp",
        "_source_file"
    )

    logger.info("Customer technical columns removed")

    display(df1)

    # ---------------------------------------------------------
    # Read Sales
    # ---------------------------------------------------------

    logger.info("Reading Silver Sales table")

    df2 = read_table("sales_clean")

    logger.info("Silver Sales table read successfully")

    logger.info(
        "Removing technical columns from Sales"
    )

    df2 = df2.drop(
        "_ingestion_timestamp",
        "_source_file"
    )

    logger.info("Sales technical columns removed")

    display(df2)

    # ---------------------------------------------------------
    # Join Customers and Sales
    # ---------------------------------------------------------

    logger.info(
        "Joining Sales with Customers using customer_id"
    )

    df = df2.join(
        df1,
        "customer_id",
        how="right"
    )

    logger.info(
        "Customers and Sales join completed"
    )

    # ---------------------------------------------------------
    # Customer Sales Aggregation
    # ---------------------------------------------------------

    logger.info(
        "Aggregating sales by customer_id and customer_name"
    )

    df_agg = (
        df
        .groupBy(
            "customer_id",
            "customer_name"
        )
        .agg(
            collect_set(
                col("order_id")
            ).alias("orders"),

            count(
                col("order_id")
            ).alias("total_orders"),

            sum(
                col("quantity")
            ).alias("total_quantity"),

            sum(
                col("unit_price")
            ).alias("total_sales")
        )
    )

    logger.info(
        "Customer sales aggregation completed"
    )
    
    logger.info('printing schema and data')
    df.printSchema()
    display(df)
    # ---------------------------------------------------------
    # Create Schema
    # ---------------------------------------------------------

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )

    # ---------------------------------------------------------
    # Save Gold Table
    # ---------------------------------------------------------

    logger.info(
        "Saving Gold Customer Sales Summary table"
    )

    save_table(
        df_agg,
        "customers_sales_summary"
    )

    logger.info(
        "Gold Customer Sales Summary table saved successfully"
    )

    logger.info(
        "Gold Customer Sales transformation completed successfully"
    )

except Exception:

    logger.exception(
        "Gold Customer Sales transformation failed"
    )

    raise